<a href="https://colab.research.google.com/github/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/blob/main/basic_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
  %env COLAB_RELEASE_TAG
except:
  pass  # Not running in colab.
else:
  %pip install --ignore-requires-python --requirement 'https://raw.githubusercontent.com/google-deepmind/concordia/main/examples/requirements.in' 'git+https://github.com/google-deepmind/concordia.git#egg=gdm-concordia'
  %pip list

In [ ]:
!pip install gdm-concordia[ollama]

Dependencies

In [ ]:
from collections.abc import Mapping
import dataclasses
from concordia.agents import entity_agent_with_logging
from concordia.associative_memory import basic_associative_memory
from concordia.components import agent as agent_components
from concordia.contrib import language_models as language_model_utils
from concordia.contrib.components.agent import situation_representation_via_narrative
from concordia.document import interactive_document
from concordia.language_model import language_model
import concordia.prefabs.entity as entity_prefabs
import concordia.prefabs.game_master as game_master_prefabs
from concordia.prefabs.simulation import generic as simulation
from concordia.typing import prefab as prefab_lib
from concordia.utils import helper_functions
from IPython import display
import numpy as np
import sentence_transformers

In [ ]:
API_TYPE = 'ollama'
MODEL_NAME = 'llama3.2:3b'
DISABLE_LANGUAGE_MODEL = False

In [ ]:
model = language_model_utils.language_model_setup(
    api_type=API_TYPE,
    model_name=MODEL_NAME,
    disable_language_model=DISABLE_LANGUAGE_MODEL,
)

In [ ]:
prefabs = {
    **helper_functions.get_package_classes(entity_prefabs),
    **helper_functions.get_package_classes(game_master_prefabs),
}

D&D Agent Prefab

In [ ]:
import dataclasses
from concordia.typing import prefab as prefab_lib
from concordia.agents import entity_agent_with_logging
from concordia.components import agent as agent_components
from concordia.contrib.components.agent import situation_representation_via_narrative

DEFAULT_INSTRUCTIONS_COMPONENT_KEY = 'Instructions'
DEFAULT_INSTRUCTIONS_PRE_ACT_LABEL = '\nInstructions'


@dataclasses.dataclass
class DNDAgent(prefab_lib.Prefab):
  """A prefab implementing an entity with a minimal set of components."""

  description: str = (
      'An entity that has a minimal set of components and is configurable by'
      ' the user. The initial set of components manage memory, observations,'
      ' and instructions. If goal is specified, the entity will have a goal '
      'constant component.'
  )
  params: Mapping[str, str] = dataclasses.field(
      default_factory=lambda: {
          'name': 'V',
          'goal': '',
          'backstory': '',
          'profession': '',
      }
  )

  def build(
      self,
      model: language_model.LanguageModel,
      memory_bank: basic_associative_memory.AssociativeMemoryBank,
  ) -> entity_agent_with_logging.EntityAgentWithLogging:
    """Build an agent.

    Args:
      model: The language model to use.
      memory_bank: The agent's memory_bank object.

    Returns:
      An entity.
    """

    agent_name = self.params.get('name', 'Bob')

    instructions = agent_components.instructions.Instructions(
        agent_name=agent_name,
        pre_act_label=DEFAULT_INSTRUCTIONS_PRE_ACT_LABEL,
    )

    observation_to_memory = agent_components.observation.ObservationToMemory()

    observation_label = '\nObservation'
    observation = agent_components.observation.LastNObservations(
        history_length=100, pre_act_label=observation_label
    )

    situation_representation_key = 'situation_representation'
    situation_representation = (
        situation_representation_via_narrative.SituationRepresentation(
            model=model,
            observation_component_key=(
                agent_components.observation.DEFAULT_OBSERVATION_COMPONENT_KEY
            ),
            declare_entity_as_protagonist=True,
        )
    )

    guiding_principle_key = 'principle'
    principle = agent_components.question_of_recent_memories.QuestionOfRecentMemories(
        model=model,
        pre_act_label=f'{agent_name} main guiding principle:',
        question=(
            f'How can {agent_name} exploit the situation for personal '
            'gain and gratification?'
        ),
        answer_prefix=f'{agent_name} understands that ',
        add_to_memory=False,
        components=[
            DEFAULT_INSTRUCTIONS_COMPONENT_KEY,
            situation_representation_key,
        ],
    )

    components_of_agent = {
        DEFAULT_INSTRUCTIONS_COMPONENT_KEY: instructions,
        'observation_to_memory': observation_to_memory,
        agent_components.observation.DEFAULT_OBSERVATION_COMPONENT_KEY: (
            observation
        ),
        agent_components.memory.DEFAULT_MEMORY_COMPONENT_KEY: (
            agent_components.memory.AssociativeMemory(memory_bank=memory_bank)
        ),
        'situation_representation': situation_representation,
        guiding_principle_key: principle,
    }

    component_order = list(components_of_agent.keys())

    act_component = agent_components.concat_act_component.ConcatActComponent(
        model=model,
        component_order=component_order,
    )

    agent = entity_agent_with_logging.EntityAgentWithLogging(
        agent_name=agent_name,
        act_component=act_component,
        context_components=components_of_agent,
    )

    return agent

In [ ]:
prefabs['dndagent__Entity'] = DNDAgent()

In [ ]:
PLACE = 'Wizards Tower Brewing Company, Forgotten Realms'
CONTEXT = ('This D&D short campaign specifically concerns a craft brewery.',
f'It is set in the {PLACE} and is in dire need of help.',
'A band of reliable, affordable adventurers are needed to sort out a RAT INFESTATION in the brewery\'s BASEMENT.',
f'For the duration of the one-shot, only Player_One, Player_Two, Player_Three and Brewery_Owner are in the brewery',
 ('At the beginning of this adventure Player_One, Player_Two and Player_Three',
f'meet in the {PLACE}. These three adventurers'),
'DO NOT know each other AT FIRST and need to get to know each other.')

NUM_STATEMENTS = 5
NAMES_TO_GENERATE = 5

prompt = interactive_document.InteractiveDocument(model)
unparsed_statements = prompt.open_question(
    question=(
         f'Generate a string of {NUM_STATEMENTS} facts about {PLACE}. '
        f'Pay special attention to {CONTEXT}. Write '
        'in present tense. Separate generated facts '
        "with ' *** '."
    ),
    max_tokens=4500,
    terminators=(),
)

statements = unparsed_statements.split('***')
statements = [n.strip() for n in statements]

for statement in statements:
  print(statement)

unparsed_names = prompt.open_question(
    question=(
        f'Generate a string of {NAMES_TO_GENERATE} names appropriate for this '
        "time and place. Include surnames. Separate them with ' *** '"
    ),
    max_tokens=4500,
    terminators=(),
)
names = unparsed_names.split('***')
names = [n.strip() for n in names]

Player_One = names[0]
Player_Two = names[1]
Player_Three = names[2]

print('\n')
print(f'Player One: {Player_One}')
print(f'Player Two: {Player_Two}')
print(f'Player Three: {Player_Three}')


prefix = f'{Player_One}, {Player_Two} and {Player_Three}'
premise = prompt.open_question(
    question=(
        f'Given the setting, why are {Player_One}, {Player_Two} and {Player_Three} about '
        'to interact?'
    ),
    answer_prefix=prefix,
)
premise = f'{prefix}{premise}'

print('\n')
print(premise)

player_one_context = prompt.open_question(
    question=(
        f'{Player_One} has a goal or interest that, if pursued, '
        f'they would need to work with {Player_Two} and {Player_Three}. What is it?'
    ),
    max_tokens=1000,
)

print('\n')
print(player_one_context)

player_two_context = prompt.open_question(
    question=(
        f'{Player_Two} has a goal or interest that, if pursued, '
        f'they would need to work with {Player_One} and {Player_Three}. What is it?'
    ),
    max_tokens=1000,
)

print('\n')
print(player_two_context)

player_three_context = prompt.open_question(
    question=(
        f'{Player_Three} has a goal or interest that, if pursued, '
        f'they would need to work with {Player_One} and {Player_Two}. What is it?'
    ),
    max_tokens=1000,
)

print('\n')
print(player_three_context)

In [ ]:
instances = [
    prefab_lib.InstanceConfig(
        prefab='dndagent__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': Player_One,
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='dndagent__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': Player_Two,
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='dndagent__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': Player_Three,
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='generic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'default rules',
            'acting_order': 'game_master_choice',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='dialogic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'conversation rules',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='formative_memories_initializer__GameMaster',
        role=prefab_lib.Role.INITIALIZER,
        params={
            'name': 'initial setup rules',
            'next_game_master_name': 'default rules',
            'shared_memories': statements,
            'player_specific_context': {
                Player_One: player_one_context,
                Player_Two: player_two_context,
                Player_Three: player_three_context,
            },
        },
    ),
]

In [ ]:
config = prefab_lib.Config(
    default_premise=premise,
    default_max_steps=5,
    prefabs=prefabs,
    instances=instances,
)

In [ ]:
DISABLE_LANGUAGE_MODEL = False
# if not DISABLE_LANGUAGE_MODEL and not API_KEY:
#   raise ValueError('API_KEY is required.')
if DISABLE_LANGUAGE_MODEL:
  embedder = lambda _: np.ones(3)
else:
  st_model = sentence_transformers.SentenceTransformer(
      'sentence-transformers/all-mpnet-base-v2'
  )
  embedder = lambda x: st_model.encode(x, show_progress_bar=False)

In [ ]:
runnable_simulation = simulation.Simulation(
    config=config,
    model=model,
    embedder=embedder,
)

In [ ]:
results_log = runnable_simulation.play(max_steps=5)